# String Extensions

Runnable samples for every method in `CSharpHelperExtensions.Strings`.  
Run the **Setup** cell first, then any section independently.

| Section | Methods |
|---|---|
| [1. Null-Safety & Coalescing](#1-null-safety--coalescing) | `IsNullOrEmpty` · `HasValue` · `OrEmpty` · `OrDefault` |
| [2. Parsing](#2-parsing) | `ToNullable<T>` · `ToIntOrNull` · `ToDecimalOrNull` · `ToDateTimeOrNull` · `ToGuidOrNull` · `ToBoolOrNull` |
| [3. Text Transformation](#3-text-transformation) | `TrimToLower` · `TrimToUpper` · `ToTitleCase` · `ToSlug` · `Reverse` · `Truncate` · `MaskStart` |
| [4. Whitespace](#4-whitespace) | `RemoveWhitespace` · `CollapseWhitespace` |
| [5. Case-Insensitive Comparisons](#5-case-insensitive-comparisons) | `EqualsIgnoreCase` · `ContainsIgnoreCase` · `StartsWithIgnoreCase` · `EndsWithIgnoreCase` |
| [6. Prefix & Suffix](#6-prefix--suffix) | `EnsurePrefix` · `EnsureSuffix` · `TrimPrefix` · `TrimSuffix` |
| [7. Splitting & Joining](#7-splitting--joining) | `SplitNonEmpty` · `JoinWith` · `ReplaceMany` |
| [8. Character Validation](#8-character-validation) | `IsNumeric` · `IsAlpha` · `IsAlphaNumeric` |
| [9. Encoding & Bytes](#9-encoding--bytes) | `Base64Encode` · `Base64Decode` · `ToBase64Url` · `FromBase64Url` · `ToUtf8Bytes` · `ToUtf8Stream` |
| [10. Internationalisation](#10-internationalisation) | `RemoveDiacritics` |
| [11. Chaining Examples](#11-chaining-examples) | Composing multiple string extensions into pipelines |

## Setup

> **Run this cell first.** It loads the compiled library and imports both namespaces.
>
> Build first if the DLL is missing: `dotnet build` from the repo root.

In [ ]:
#r "../src/CSharpHelperExtensions/bin/Debug/net10.0/CSharpHelperExtensions.dll"
using CSharpHelperExtensions;         // IsNullOrEmpty, In, IsBetween, ToJson
using CSharpHelperExtensions.Strings; // all StringExtensions

---
## 1. Null-Safety & Coalescing

| Method | Signature | Returns |
|---|---|---|
| `IsNullOrEmpty` | `string → bool` | `true` when null, empty, or whitespace-only |
| `HasValue` | `string → bool` | `true` when not null/whitespace |
| `OrEmpty` | `string → string` | `""` when null; passes whitespace through |
| `OrDefault` | `string, string → string` | fallback when null **or** whitespace |

In [4]:
// IsNullOrEmpty — true for null, empty, or whitespace-only strings
display(((string)null).IsNullOrEmpty());  // True
display("".IsNullOrEmpty());              // True
display("   ".IsNullOrEmpty());           // True  (whitespace counts as empty)
display("hello".IsNullOrEmpty());         // False
display("  x  ".IsNullOrEmpty());         // False  (has real content)

True

True

True

False

False

In [5]:
// HasValue — true only when there is real content
display("hello".HasValue());            // True
display("  ".HasValue());               // False  (whitespace only)
display(((string)null).HasValue());     // False

True

False

False

In [6]:
// OrEmpty — safe null-to-empty coalesce; whitespace strings pass through unchanged
display(((string)null).OrEmpty());      // ""
display("  ".OrEmpty());               // "  "  ← whitespace preserved
display("hello".OrEmpty());            // "hello"

hello

In [7]:
// OrDefault — replaces null AND whitespace with a fallback value
display(((string)null).OrDefault("N/A"));   // "N/A"
display("   ".OrDefault("N/A"));            // "N/A"  ← whitespace → fallback
display("hello".OrDefault("N/A"));          // "hello"

N/A

N/A

hello

---
## 2. Parsing

Two flavours:
- **`ToNullable<T>`** — generic, uses `TypeConverter`; throws on bad format
- **`ToXxxOrNull`** — type-specific, returns `null` on bad format (never throws)

All methods return `null` for `null`, empty, or whitespace input.

In [8]:
// ToNullable<T> — generic converter; throws FormatException on invalid input
display("42".ToNullable<int>());                     // 42
display("".ToNullable<int>());                        // null
display("   ".ToNullable<int>());                    // null  (whitespace)
display("2024-06-15".ToNullable<DateTime>());        // 15/06/2024 ...
display("3fa85f64-5717-4562-b3fc-2c963f66afa6".ToNullable<Guid>()); // Guid

42

<null>

<null>

2024-06-15 00:00:00Z

3fa85f64-5717-4562-b3fc-2c963f66afa6

In [9]:
// ToIntOrNull / ToDecimalOrNull — invariant-culture, no exceptions
display("99".ToIntOrNull());            // 99
display("3.14".ToIntOrNull());          // null  (not an int)
display("abc".ToIntOrNull());           // null

display("3.14".ToDecimalOrNull());      // 3.14
display("1,000.50".ToDecimalOrNull());  // 1000.50  (comma as thousands separator)
display("xyz".ToDecimalOrNull());       // null

99

<null>

<null>

3.14

1000.50

<null>

In [10]:
// ToDateTimeOrNull / ToGuidOrNull / ToBoolOrNull
display("2024-01-15".ToDateTimeOrNull());                         // DateTime
display("not-a-date".ToDateTimeOrNull());                         // null

display("3fa85f64-5717-4562-b3fc-2c963f66afa6".ToGuidOrNull());  // Guid
display("bad-guid".ToGuidOrNull());                               // null

display("true".ToBoolOrNull());    // True
display("False".ToBoolOrNull());   // False  (case-insensitive)
display("yes".ToBoolOrNull());     // null   (only "true"/"false" accepted)

2024-01-15 00:00:00Z

<null>

3fa85f64-5717-4562-b3fc-2c963f66afa6

<null>

True

False

<null>

---
## 3. Text Transformation

| Method | What it does |
|---|---|
| `TrimToLower` | trim + lowercase (invariant) |
| `TrimToUpper` | trim + uppercase (invariant) |
| `ToTitleCase` | capitalize first letter of each word |
| `ToSlug` | URL-friendly lowercase slug |
| `Reverse` | reverse characters |
| `Truncate` | keep first N characters |
| `MaskStart` | mask all but last N characters |

In [11]:
// TrimToLower / TrimToUpper
display("  Hello World  ".TrimToLower());   // "hello world"
display("  Hello World  ".TrimToUpper());   // "HELLO WORLD"
display(((string)null).TrimToLower());      // ""  (null-safe)

hello world

HELLO WORLD

In [12]:
// ToTitleCase — collapses whitespace, then capitalizes each word
display("hello world".ToTitleCase());           // "Hello World"
display("HELLO WORLD".ToTitleCase());           // "Hello World"  (lowercases first)
display("  the   quick brown  fox  ".ToTitleCase()); // "The Quick Brown Fox"
display("".ToTitleCase());                       // ""

Hello World

Hello World

The Quick Brown Fox

In [13]:
// ToSlug — URL-safe: lowercase, diacritics removed, non-alphanumeric → single dash
display("Hello World!".ToSlug());               // "hello-world"
display("  C# Helper Extensions  ".ToSlug());   // "c-helper-extensions"
display("café au lait".ToSlug());               // "cafe-au-lait"
display("---leading dashes---".ToSlug());       // "leading-dashes"

hello-world

c-helper-extensions

cafe-au-lait

leading-dashes

In [14]:
// Reverse
display("hello".Reverse());        // "olleh"
display("abcde".Reverse());        // "edcba"
display("".Reverse());             // ""
display(((string)null).Reverse()); // ""

olleh

edcba

In [15]:
// Truncate — keep first maxLength characters
display("Hello, World!".Truncate(5));    // "Hello"
display("Hi".Truncate(10));             // "Hi"  (shorter than limit)
display("Hello".Truncate(5));           // "Hello"  (exact length)
display(((string)null).Truncate(5));    // ""

Hello

Hi

Hello

In [16]:
// MaskStart — hide all but the last N characters (useful for credentials/card numbers)
display("4111111111111234".MaskStart(4));       // "************1234"
display("4111111111111234".MaskStart(4, '#'));  // "############1234"  (custom mask char)
display("AB".MaskStart(4));                    // "AB"  (visible > length, nothing masked)
display(((string)null).MaskStart(4));          // ""

************1234

############1234

AB

---
## 4. Whitespace

| Method | What it does |
|---|---|
| `RemoveWhitespace` | strips **all** whitespace characters |
| `CollapseWhitespace` | trims edges, collapses internal runs to a single space |

In [17]:
// RemoveWhitespace — removes every whitespace character (spaces, tabs, newlines)
display("hello world".RemoveWhitespace());       // "helloworld"
display("  h e l l o  ".RemoveWhitespace());    // "hello"
display("a\t b\n c".RemoveWhitespace());        // "abc"
display(((string)null).RemoveWhitespace());      // ""

helloworld

hello

abc

In [18]:
// CollapseWhitespace — trim + collapse internal runs to a single space
display("  hello   world  ".CollapseWhitespace());      // "hello world"
display("one\t\ttwo\n\nthree".CollapseWhitespace());   // "one two three"
display("already clean".CollapseWhitespace());          // "already clean"
display("   ".CollapseWhitespace());                    // ""

hello world

one two three

already clean

---
## 5. Case-Insensitive Comparisons

All four methods use **ordinal case-insensitive** comparison and are null-safe on both sides.

In [19]:
// EqualsIgnoreCase
display("Hello".EqualsIgnoreCase("hello"));   // True
display("Hello".EqualsIgnoreCase("HELLO"));   // True
display("Hello".EqualsIgnoreCase("world"));   // False

True

True

False

In [20]:
// ContainsIgnoreCase
display("Hello, World!".ContainsIgnoreCase("world"));   // True
display("Hello, World!".ContainsIgnoreCase("HELLO"));   // True
display("Hello, World!".ContainsIgnoreCase("xyz"));     // False
display(((string)null).ContainsIgnoreCase("hello"));    // False  (null-safe)

True

True

False

False

In [21]:
// StartsWithIgnoreCase / EndsWithIgnoreCase
display("Hello, World!".StartsWithIgnoreCase("hello"));  // True
display("Hello, World!".StartsWithIgnoreCase("WORLD"));  // False

display("Hello, World!".EndsWithIgnoreCase("world!"));   // True
display("Hello, World!".EndsWithIgnoreCase("HELLO"));    // False

True

False

True

False

---
## 6. Prefix & Suffix

| Method | What it does |
|---|---|
| `EnsurePrefix` | prepends if not already present |
| `EnsureSuffix` | appends if not already present |
| `TrimPrefix` | removes prefix if present |
| `TrimSuffix` | removes suffix if present |

In [22]:
// EnsurePrefix — idempotent: won't double-add
display("/api/users".EnsurePrefix("/"));     // "/api/users"  (already has it)
display("api/users".EnsurePrefix("/"));      // "/api/users"  (added)
display(((string)null).EnsurePrefix("/"));   // "/"

/api/users

/api/users

/

In [23]:
// EnsureSuffix — idempotent: won't double-add
display("https://example.com/".EnsureSuffix("/"));   // "https://example.com/"  (already has it)
display("https://example.com".EnsureSuffix("/"));    // "https://example.com/"  (added)
display(((string)null).EnsureSuffix("/"));            // "/"

https://example.com/

https://example.com/

/

In [24]:
// TrimPrefix / TrimSuffix — removes exactly one occurrence if present
display("/api/users".TrimPrefix("/"));      // "api/users"
display("api/users".TrimPrefix("/"));       // "api/users"  (no prefix, unchanged)

display("report.csv".TrimSuffix(".csv"));  // "report"
display("report.txt".TrimSuffix(".csv"));  // "report.txt"  (no suffix, unchanged)

api/users

api/users

report

report.txt

---
## 7. Splitting & Joining

| Method | What it does |
|---|---|
| `SplitNonEmpty` | splits and discards empty segments |
| `JoinWith` | fluent `string.Join` (separator is `this`) |
| `ReplaceMany` | apply a sequence of find-and-replace pairs |

In [25]:
// SplitNonEmpty — splits on one or more separators, drops empty entries
display("a,b,,c,".SplitNonEmpty(','));          // ["a", "b", "c"]
display("one|two||three".SplitNonEmpty('|'));   // ["one", "two", "three"]
display("a,b;c".SplitNonEmpty(',', ';'));       // ["a", "b", "c"]  (multiple separators)
display(((string)null).SplitNonEmpty(','));     // []  (null-safe)

[ a, b, c ]

[ one, two, three ]

[ a, b, c ]

[ ]

In [26]:
// JoinWith — separator is the receiver (reads naturally in pipeline style)
var words = new[] { "one", "two", "three" };

display(", ".JoinWith(words));       // "one, two, three"
display(" | ".JoinWith(words));      // "one | two | three"
display("".JoinWith(words));         // "onetwothree"

one, two, three

one | two | three

onetwothree

In [27]:
// ReplaceMany — ordered replacements applied sequentially
var replacements = new (string, string)[] { ("foo", "bar"), ("baz", "qux") };
display("foo and baz".ReplaceMany(replacements));   // "bar and qux"

// Practical use: sanitise user input
var htmlEscape = new (string, string)[] { ("&", "&amp;"), ("<", "&lt;"), (">", "&gt;") };
display("<b>Hello & world</b>".ReplaceMany(htmlEscape));  // "&lt;b&gt;Hello &amp; world&lt;/b&gt;"

bar and qux

&lt;b&gt;Hello &amp; world&lt;/b&gt;

---
## 8. Character Validation

All three methods return `false` for null/empty strings.

In [28]:
// IsNumeric — all characters are digits 0–9
display("12345".IsNumeric());     // True
display("123.45".IsNumeric());    // False  (dot is not a digit)
display("-5".IsNumeric());        // False  (minus sign)
display("".IsNumeric());          // False

True

False

False

False

In [29]:
// IsAlpha — all characters are Unicode letters
display("Hello".IsAlpha());       // True
display("Hello123".IsAlpha());    // False
display("café".IsAlpha());        // True  (accented letters are letters)

True

False

True

In [30]:
// IsAlphaNumeric — all characters are letters or digits
display("Hello123".IsAlphaNumeric());    // True
display("Hello 123".IsAlphaNumeric());   // False  (space)
display("abc!".IsAlphaNumeric());        // False  (punctuation)

True

False

False

---
## 9. Encoding & Bytes

| Method pair | Notes |
|---|---|
| `Base64Encode` / `Base64Decode` | standard Base64 (may contain `+`, `/`, `=`) |
| `ToBase64Url` / `FromBase64Url` | URL-safe Base64 — no `+`, `/`, or `=` padding |
| `ToUtf8Bytes` | UTF-8 `byte[]` |
| `ToUtf8Stream` | seekable `MemoryStream` |

In [31]:
// Standard Base64 — roundtrip
var encoded = "Hello, World!".Base64Encode();
display(encoded);                    // "SGVsbG8sIFdvcmxkIQ=="
display(encoded.Base64Decode());     // "Hello, World!"

display(((string)null).Base64Encode()); // null  (null-safe)

SGVsbG8sIFdvcmxkIQ==

Hello, World!

<null>

In [32]:
// URL-safe Base64 — no +, /, or = characters; safe for query strings / JWT tokens
var urlToken = "Hello, World!".ToBase64Url();
display(urlToken);                    // "SGVsbG8sIFdvcmxkIQ"  (no padding)
display(urlToken.FromBase64Url());    // "Hello, World!"

// Compare standard vs URL-safe
var sample = "hello+world/test=";
display(sample.Base64Encode());      // contains + and / and =
display(sample.ToBase64Url());       // replaced with - and _ , no =

SGVsbG8sIFdvcmxkIQ

Hello, World!

aGVsbG8rd29ybGQvdGVzdD0=

aGVsbG8rd29ybGQvdGVzdD0

In [33]:
// ToUtf8Bytes — get raw UTF-8 bytes
var bytes = "hello".ToUtf8Bytes();
display(bytes.Length);                                   // 5
display(string.Join("-", bytes.Select(b => b.ToString("X2")))); // "68-65-6C-6C-6F"

// ToUtf8Stream — get a seekable MemoryStream (useful for APIs expecting a Stream)
using var stream = "hello".ToUtf8Stream();
display(stream.Length);                                  // 5
display(new System.IO.StreamReader(stream).ReadToEnd()); // "hello"


(7,11): error CS1002: ; expected



Error: compilation error

---
## 10. Internationalisation

`RemoveDiacritics` strips accent marks (non-spacing marks) using Unicode normalisation.  
Useful before slug generation, search indexing, or loose string comparison.

In [ ]:
// RemoveDiacritics
display("café".RemoveDiacritics());          // "cafe"
display("naïve".RemoveDiacritics());         // "naive"
display("Ångström".RemoveDiacritics());      // "Angstrom"
display("São Paulo".RemoveDiacritics());     // "Sao Paulo"
display("über cool".RemoveDiacritics());     // "uber cool"

// Common pattern: normalise before comparison
var search = "cafe";
var value  = "café";
display(value.RemoveDiacritics().EqualsIgnoreCase(search));  // True

---
## 11. Chaining Examples

String extensions are designed to compose. These pipelines show how to chain methods
for common real-world scenarios.


### Defensive input normalisation
`OrDefault → CollapseWhitespace → TrimToLower`

In [ ]:
// Normalise untrusted user input before storing or comparing
string? userInput = "  Hello   WORLD  ";

var normalised = userInput
    .OrDefault("unknown")    // replace null/whitespace with fallback
    .CollapseWhitespace()   // "Hello WORLD"
    .TrimToLower();         // "hello world"

display(normalised);        // "hello world"

// Works safely when input is null or blank
display(((string)null).OrDefault("unknown").CollapseWhitespace().TrimToLower());
// "unknown"

### URL slug generation from raw titles
`OrEmpty → CollapseWhitespace → RemoveDiacritics → ToSlug → EnsurePrefix`

In [ ]:
string? title = "  Ångström & Café: The Story  ";

var slug = title
    .OrEmpty()              // null → ""
    .CollapseWhitespace()   // "Ångström & Café: The Story"
    .RemoveDiacritics()     // "Angstrom & Cafe: The Story"
    .ToSlug()               // "angstrom-cafe-the-story"
    .EnsurePrefix("/");     // "/angstrom-cafe-the-story"

display(slug);              // "/angstrom-cafe-the-story"

### Safe display name with truncation
`OrDefault → ToTitleCase → Truncate → EnsureSuffix`

In [ ]:
string? rawName = "  alice smith  ";

var displayName = rawName
    .OrDefault("Anonymous") // fallback for null/blank
    .ToTitleCase()          // "Alice Smith"
    .Truncate(10)           // "Alice Smit" (if too long)
    .EnsureSuffix(".");     // "Alice Smit."

display(displayName);       // "Alice Smit."

display(((string)null).OrDefault("Anonymous").ToTitleCase().Truncate(10));
// "Anonymous"

### Credential masking pipeline
`OrEmpty → TrimToLower → MaskStart`

In [ ]:
// Display an API key or card number safely
string? apiKey = "  sk-ABCDEF1234567890  ";

var safe = apiKey
    .OrEmpty()
    .TrimToLower()          // "sk-abcdef1234567890"
    .MaskStart(4);          // "**************7890"

display(safe);              // "**************7890"
display(((string)null).OrEmpty().MaskStart(4)); // ""  (null-safe)

### CSV tag parsing and normalisation
`OrEmpty → SplitNonEmpty → Select(TrimToLower) → JoinWith`

In [ ]:
using CSharpHelperExtensions.Strings;

string? csv = "  Tech ,  SCIENCE , , art  , Tech ";

var tags = csv
    .OrEmpty()
    .SplitNonEmpty(',')                         // ["  Tech ", "  SCIENCE ", " art  ", " Tech "]
    .Select(t => t.TrimToLower())               // ["tech", "science", "art", "tech"]
    .Distinct()                                  // ["tech", "science", "art"]
    .OrderBy(t => t)                             // ["art", "science", "tech"]
    .ToList();

display(", ".JoinWith(tags));   // "art, science, tech"

### Search normalisation for diacritic-insensitive lookup
`RemoveDiacritics → TrimToLower → ContainsIgnoreCase`

In [ ]:
// Search a name list where entries may have accents
var names = new[] { "José García", "André Martin", "John Smith", "Ångström Lab" };
string? query = "  garcia  ";

var normQuery = query
    .OrEmpty()
    .RemoveDiacritics()
    .TrimToLower();   // "garcia"

var matches = names
    .Where(n => n.RemoveDiacritics().ContainsIgnoreCase(normQuery))
    .ToList();

display(matches);   // ["José García"]

### Dynamic base-URL construction
`OrDefault → TrimSuffix → EnsureSuffix → EnsurePrefix`

In [ ]:
string? baseUrl = "  https://api.example.com/  ";
string? path    = "v2/users";

var url = baseUrl
    .OrDefault("https://localhost")
    .CollapseWhitespace()     // strip internal whitespace too
    .TrimSuffix("/")          // remove trailing slash
    + "/" +
    path
    .OrEmpty()
    .TrimPrefix("/");         // remove leading slash

display(url);   // "https://api.example.com/v2/users"